<a href="https://colab.research.google.com/github/puskat-debug/database-assign.ipynb/blob/main/Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pymongo dnspython --quiet

%load_ext rpy2.ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.7 MB/s eta 0:00:00


In [7]:
%%R
install.packages(c("sqldf", "dplyr", "lubridate", "ggplot2", "corrplot", "tidyr"),
                 repos = "https://cran.r-project.org", quiet = TRUE)

In [11]:
from google.colab import files
uploaded = files.upload()

Saving app_events.csv to app_events.csv
Saving complaints.csv to complaints.csv
Saving customers.csv to customers.csv
Saving data_dictionary.csv to data_dictionary.csv
Saving deliveries.csv to deliveries.csv
Saving drivers.csv to drivers.csv
Saving hubs.csv to hubs.csv
Saving incidents.csv to incidents.csv
Saving northstar_cleaned_app_events.csv to northstar_cleaned_app_events.csv
Saving northstar_data_dictionary.csv to northstar_data_dictionary.csv
Saving orders.csv to orders.csv
Saving vehicles.csv to vehicles.csv


In [12]:
print("Files uploaded successfully:")
csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
for i, f in enumerate(csv_files, 1):
    print(f"{i}. {f}")
print(f"\nTotal CSV files: {len(csv_files)}")

Files uploaded successfully:
1. app_events.csv
2. complaints.csv
3. customers.csv
4. data_dictionary.csv
5. deliveries.csv
6. drivers.csv
7. hubs.csv
8. incidents.csv
9. northstar_cleaned_app_events.csv
10. northstar_data_dictionary.csv
11. orders.csv
12. vehicles.csv

Total CSV files: 12


In [13]:
import pandas as pd
import io

for filename in csv_files:
    print(f"\n{'='*60}")
    print(f"{filename}")
    print(f"{'='*60}")

    df_temp = pd.read_csv(io.BytesIO(uploaded[filename]))

    print(f"Shape: {df_temp.shape}")
    print("Columns:", df_temp.columns.tolist())
    print("\nFirst 3 rows:")
    print(df_temp.head(3))
    print("\nMissing values per column:")
    print(df_temp.isnull().sum())
    print("-" * 40)


app_events.csv
Shape: (640, 10)
Columns: ['event_id', 'customer_id', 'order_id', 'event_timestamp', 'event_type', 'session_id', 'device_type', 'zone_context', 'api_latency_ms', 'success_flag']

First 3 rows:
  event_id customer_id order_id      event_timestamp    event_type session_id  \
0  AE00001       C0488      NaN  2024-08-09 03:25:00   eta_refresh     S19847   
1  AE00002       C0595   O00950  2024-02-13 22:29:00  search_route     S32766   
2  AE00003       C0494   O00170  2025-08-11 09:29:00   chat_opened     S99516   

  device_type zone_context  api_latency_ms  success_flag  
0     Android        north             301             1  
1     Android        SOUTH              60             1  
2         iOS      Airport            1118             1  

Missing values per column:
event_id             0
customer_id          0
order_id           144
event_timestamp      0
event_type           0
session_id           0
device_type          0
zone_context         0
api_latency_ms    

In [14]:
cleaned_dfs = {}

for filename in csv_files:
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    df.columns = df.columns.str.strip()
    df = df.dropna(how='all')
    df = df.drop_duplicates()

    cleaned_dfs[filename] = df
    print(f"Cleaned {filename}: {df.shape}")

Cleaned app_events.csv: (640, 10)
Cleaned complaints.csv: (320, 10)
Cleaned customers.csv: (650, 9)
Cleaned data_dictionary.csv: (9, 3)
Cleaned deliveries.csv: (950, 13)
Cleaned drivers.csv: (170, 8)
Cleaned hubs.csv: (8, 5)
Cleaned incidents.csv: (280, 7)
Cleaned northstar_cleaned_app_events.csv: (640, 10)
Cleaned northstar_data_dictionary.csv: (9, 3)
Cleaned orders.csv: (1250, 11)
Cleaned vehicles.csv: (120, 8)


In [15]:
customers  = cleaned_dfs['customers.csv']
drivers    = cleaned_dfs['drivers.csv']
vehicles   = cleaned_dfs['vehicles.csv']
hubs       = cleaned_dfs['hubs.csv']
orders     = cleaned_dfs['orders.csv']
deliveries = cleaned_dfs['deliveries.csv']
app_events = cleaned_dfs['app_events.csv']
incidents  = cleaned_dfs['incidents.csv']
complaints = cleaned_dfs['complaints.csv']

print("Master data:")
print(f"  Customers : {customers.shape}")
print(f"  Drivers   : {drivers.shape}")
print(f"  Vehicles  : {vehicles.shape}")
print(f"  Hubs      : {hubs.shape}")
print("\nTransaction data:")
print(f"  Orders     : {orders.shape}")
print(f"  Deliveries : {deliveries.shape}")
print("\nEvent data:")
print(f"  App events : {app_events.shape}")
print(f"  Incidents  : {incidents.shape}")
print(f"  Complaints : {complaints.shape}")

Master data:
  Customers : (650, 9)
  Drivers   : (170, 8)
  Vehicles  : (120, 8)
  Hubs      : (8, 5)

Transaction data:
  Orders     : (1250, 11)
  Deliveries : (950, 13)

Event data:
  App events : (640, 10)
  Incidents  : (280, 7)
  Complaints : (320, 10)


In [16]:
zone_map = {
    "ctr": "Central",   "CENTRAL": "Central",   "central": "Central",
    "NORTH": "North",   "north": "North",
    "SOUTH": "South",   "south": "South",
    "EAST":  "East",    "east":  "East",
    "WEST":  "West",    "west":  "West",
    "RiverSide": "Riverside", "RIVERSIDE": "Riverside", "riverside": "Riverside",
    "AIRPORT": "Airport",     "airport": "Airport"
}
def norm_zone(series):
    return series.str.strip().replace(zone_map)

orders["pickup_zone"]      = norm_zone(orders["pickup_zone"])
orders["dropoff_zone"]     = norm_zone(orders["dropoff_zone"])
drivers["base_zone"]       = norm_zone(drivers["base_zone"])
vehicles["assigned_zone"]  = norm_zone(vehicles["assigned_zone"])
app_events["zone_context"] = norm_zone(app_events["zone_context"])

print("Canonical zones after normalisation:", sorted(orders["pickup_zone"].unique()))

Canonical zones after normalisation: ['Airport', 'Central', 'Ctr', 'East', 'North', 'Riverside', 'South', 'West']


In [17]:
for df, col in [(deliveries, "dispatch_time"),
                (deliveries, "delivery_completed_at"),
                (orders,     "order_created_at"),
                (complaints, "created_at")]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

missing_ts = deliveries["delivery_completed_at"].isna().sum()
print(f"Deliveries with missing completion timestamp: {missing_ts}")

Deliveries with missing completion timestamp: 19
